# 03 · Selección de features + Modelamiento + Evaluación

**Proyecto:** ChurnLens · Diplomado MLDS — Universidad Nacional de Colombia  
**Fase TDSP:** 3 · *Modelamiento y extracción de características* (10 %)

**Objetivo de este notebook:** reproducir end-to-end la entrega de Fase 3 — selección de features con cuatro técnicas, comparación de baselines + modelos no lineales, y evaluación del ganador con _threshold tuning_. Toda la lógica vive en `src/churnlens/features/selection.py` y `src/churnlens/models/*`; aquí solo orquestamos y mostramos resultados.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd

from churnlens.config import settings
from churnlens.features.selection import persist_feature_selection, run_feature_selection
from churnlens.models.evaluation import (
    binary_metrics,
    optimal_threshold,
    plot_calibration,
    plot_confusion_matrix,
    plot_pr_curves,
    plot_roc_curves,
    plot_threshold_sweep,
    threshold_sweep,
)
from churnlens.models.registry import best_model_by, list_models, load_model
from churnlens.models.train import train_models
from churnlens.models.train import _predict_proba

print("settings.random_seed =", settings.random_seed)
print("processed_dir       =", settings.processed_dir)

## 1. Carga de los splits preprocesados (Fase 2)

Los tres parquet ya tienen 35 features transformadas + el target `Churn` binarizado a `int8`.

In [ ]:
train = pd.read_parquet(settings.processed_dir / "train.parquet")
val = pd.read_parquet(settings.processed_dir / "val.parquet")
test = pd.read_parquet(settings.processed_dir / "test.parquet")
for name, df in {"train": train, "val": val, "test": test}.items():
    rate = df["Churn"].mean()
    print(f"{name:5s} · shape={df.shape}  · churn_rate={rate:.4f}")

## 2. Selección de features con cuatro técnicas

Mutual Information + χ² (filtros) + L1 Logistic (embedded) + Permutation Importance sobre Random Forest (wrapper). Se consolida un top-20 por consenso. Detalle en [`docs/modeling/feature_selection.md`](../docs/modeling/feature_selection.md).

In [ ]:
selection = run_feature_selection(k=20, random_state=settings.random_seed)
persist_feature_selection(selection)
selection.consensus.head(20)

In [ ]:
print("Top-20 por consenso:")
for i, feat in enumerate(selection.top_k, start=1):
    print(f"  {i:2d}. {feat}")

## 3. Entrenamiento de los 8 modelos del catálogo

Se entrenan, en orden: 3 _dummies_ (referencia trivial), Logistic Regression (L2 balanced y L1 esparsa), Random Forest, HistGradientBoosting y LightGBM. Cada modelo se evalúa por validación cruzada estratificada 5-fold (PR-AUC + ROC-AUC) y luego se ajusta sobre todo `train`.

In [ ]:
artifacts = train_models(cv=5)
print(f"Ganador por PR-AUC en val: {artifacts.best_model_name}")
artifacts.summary_table

In [ ]:
artifacts.cv_table

## 4. Evaluación del modelo ganador

Se cargan estimador + manifiesto desde el registro local, se predicen probabilidades sobre `val` y se sintoniza el _threshold_ maximizando F1. Las figuras se persisten en `reports/figures/`.

In [ ]:
entry = best_model_by("pr_auc", split="val")
assert entry is not None
model, metadata = load_model(entry.name)
feature_set = metadata["feature_set"]
y_val = val["Churn"].to_numpy()
x_val = val[feature_set].astype("float32").to_numpy()
proba_val = _predict_proba(model, x_val)

choice = optimal_threshold(y_val, proba_val, metric="f1")
metrics_default = binary_metrics(y_val, proba_val, threshold=0.5)
metrics_tuned = binary_metrics(y_val, proba_val, threshold=choice.threshold)
print("Modelo:", entry.name)
print("Threshold óptimo (F1):", choice.threshold)
print("\nMétricas @ thr=0.5")
for k, v in metrics_default.items():
    print(f"  {k:18s} {v:.4f}")
print("\nMétricas @ thr sintonizado")
for k, v in metrics_tuned.items():
    print(f"  {k:18s} {v:.4f}")

In [ ]:
sweep = threshold_sweep(y_val, proba_val)
sweep.loc[[0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]]

In [ ]:
figures_dir = PROJECT_ROOT / "reports" / "figures"
plot_pr_curves({entry.name: (y_val, proba_val)}, out_path=figures_dir / f"nb_pr_{entry.name}.png")
plot_roc_curves({entry.name: (y_val, proba_val)}, out_path=figures_dir / f"nb_roc_{entry.name}.png")
plot_calibration(y_val, proba_val, out_path=figures_dir / f"nb_calibration_{entry.name}.png")
plot_threshold_sweep(
    sweep,
    out_path=figures_dir / f"nb_threshold_{entry.name}.png",
    chosen=choice.threshold,
)
plot_confusion_matrix(
    y_val,
    (proba_val >= choice.threshold).astype("int8"),
    out_path=figures_dir / f"nb_confusion_{entry.name}.png",
)

## 5. Modelos registrados localmente

In [ ]:
rows = []
for e in list_models():
    val_m = (e.metrics.get("val") or {})
    tuned_m = (e.metrics.get("val_tuned") or {})
    rows.append({
        "name": e.name,
        "algorithm": e.algorithm,
        "val_pr_auc": val_m.get("pr_auc"),
        "val_roc_auc": val_m.get("roc_auc"),
        "val_f1_tuned": tuned_m.get("f1"),
        "created_at": e.created_at,
    })
pd.DataFrame(rows).sort_values("val_pr_auc", ascending=False)

## 6. Conclusiones operativas

El reporte narrativo (decisiones, justificaciones, _next steps_ de Fase 4) vive en:

* [`docs/modeling/feature_selection.md`](../docs/modeling/feature_selection.md)
* [`docs/modeling/baseline_models.md`](../docs/modeling/baseline_models.md)
* [`docs/modeling/final_model_report.md`](../docs/modeling/final_model_report.md)

El _held-out_ `test.parquet` **no se evalúa aquí** — está reservado para la Fase 4.